This notebook is used to synthesize the trained models with HLS4MLs backend Vitis Unified. Some synthesis include bitfile-generation, and varies with strategy.

In [23]:
import os
import sys
import time
import numpy as np
import matplotlib.pyplot as plt
import keras
import tensorflow as tf
from sklearn.metrics import accuracy_score

# Load Vitis into path
os.environ['PATH'] = os.environ['XILINX_VITIS'] + '/bin:' + os.environ['PATH']

In [ ]:
model_to_test = 'pixsplit_hgq2'

model_configs = [
    {
        "description": "AdaptiveHP acc=0.7084 ebops=864 Distributed Arithmetic",
        "model_revision": "Training_AdaptiveHP",
        "keras_model_path": "pixsplit_hgq2/Training_AdaptiveHP/model_Training_AdaptiveHP_acc=0.7084_ebops=864.keras",
        "hls4ml_strategy": "DA",
        "hls4ml_generate_bitfile": True,
        "hls4ml_revision": "acc=0.7084_ebops=864_VU_DA_bitfile",
    },
    #{
    #    "description": "AdaptiveHP acc=0.7084 ebops=864 latency",
    #    "model_revision": "Training_AdaptiveHP",
    #    "keras_model_path": "pixsplit_hgq2/Training_AdaptiveHP/model_Training_AdaptiveHP_acc=0.7084_ebops=864.keras",
    #    "hls4ml_strategy": "latency",
    #    "hls4ml_generate_bitfile": True,
    #    "hls4ml_revision": "acc=0.7084_ebops=864_VU_latency_bitfile",
    #},
    
    {
        "description": "AdaptiveHP acc=0.7449 ebops=3640 Distributed Arithmetic",
        "model_revision": "Training_AdaptiveHP",
        "keras_model_path": "pixsplit_hgq2/Training_AdaptiveHP/model_Training_AdaptiveHP_acc=0.7449_ebops=3640.keras",
        "hls4ml_strategy": "DA",
        "hls4ml_generate_bitfile": True,
        "hls4ml_revision": "acc=0.7449_ebops=3640_VU_DA_bitfile",
    },
    
    {
        "description": "AdaptiveHP acc=0.7573 ebops=11698 Distributed Arithmetic",
        "model_revision": "Training_AdaptiveHP",
        "keras_model_path": "pixsplit_hgq2/Training_AdaptiveHP/model_Training_AdaptiveHP_acc=0.7573_ebops=11698.keras",
        "hls4ml_strategy": "DA",
        "hls4ml_generate_bitfile": True,
        "hls4ml_revision": "acc=0.7573_ebops=11698_VU_DA_bitfile",
    },
    #{
    #    "description": "AdaptiveHP acc=0.7573 ebops=11698 latency",
    #    "model_revision": "Training_AdaptiveHP",
    #    "keras_model_path": "pixsplit_hgq2/Training_AdaptiveHP/model_Training_AdaptiveHP_acc=0.7573_ebops=11698.keras",
    #    "hls4ml_strategy": "latency",
    #    "hls4ml_generate_bitfile": True,
    #    "hls4ml_revision": "acc=0.7573_ebops=11698_VU_latency_bitfile",
    #},
]

In [34]:
# Load dataset which is preprocessed in another notebook
# Used for calibration (x) and simulation, which we do not do here

#X_train = np.load("Data/processed_data/X_train.npy")
#X_val = np.load("Data/processed_data/X_val.npy")
x_test = np.load('Data/processed_data/X_test.npy') # used for calibrating
#y_train = np.load("Data/processed_data/y_train.npy")
#y_val = np.load("Data/processed_data/y_val.npy")
y_test = np.load("Data/processed_data/y_test.npy")

x_test = x_test.astype('float32')
x_test.dtype

dtype('float32')

In [26]:
import os

def prepare_directory(model_config):
    output_dir = os.path.join(
        os.path.dirname(os.path.abspath(model_config["keras_model_path"])),
        f"hls4ml_prj_{model_config['hls4ml_revision']}",
    )
    os.makedirs(output_dir, exist_ok=True)

    description = f"""
    Description of HLS4ML-project.

    {model_config['description']}

    - Bitfile: {model_config['hls4ml_generate_bitfile']}
    - Environment: devenv-hgq+da (environment-HGQ+DA.yml)
    - Target Device: KV260 (xck26-sfvc784-2LV-c)
    - Dataset: Pixel Cluster Splitting
    - Vivado/Vitis: 2025.2
    - Model Architecture: {model_to_test}
    - Model Revision: {model_config['model_revision']}
    - HLS4ML Revision: {model_config['hls4ml_revision']}

    The model summary is in the parent-directory, `summary.txt`
    """
    with open(os.path.join(output_dir, "description.md"), "w", encoding="utf-8") as f:
        f.write(description)

    return output_dir

In [ ]:
from keras.models import load_model
import hgq.layers
import hls4ml
from hgq.utils import trace_minmax

def compile_model(keras_model_path, output_dir, hls4ml_strategy):
    model = load_model(keras_model_path)
    
    # Calibrate datalane in HGQ2-model since it has layers with WRAP
    trace_minmax(model, x_test, verbose=True)
    
    hls_config = hls4ml.utils.config_from_keras_model(model, granularity='name')
    
    strategy = 'Distributed Arithmetic' if hls4ml_strategy == 'DA' else hls4ml_strategy
    hls_config['Model']['Strategy'] = strategy # https://fastmachinelearning.org/hls4ml/api/configuration.html#top-level-configuration

    hls_model = hls4ml.converters.convert_from_keras_model( 
        model,    
        backend     =   'vitisunified',
        hls_config  =   hls_config,
        output_dir  =   output_dir, 
        board       =   'kv260',
        part        =   'xck26-sfvc784-2LV-c',
        in_stream_buf_size = 512,
        clock_period=   '5',
    )
    hls_model.compile()
    return hls_model

In [31]:
def evaluate_model(hls_model,keras_model_path):
    model = load_model(keras_model_path)
    #y_keras = model.predict(x_test)
    y_hls = hls_model.predict(x_test.astype('float32'))
    #y_hls = np.argmax(y_hls,axis=1)
    
    from sklearn.metrics import accuracy_score
    #return accuracy_score(np.argmax(y_test, axis=1), np.argmax(y_hls, axis=1)) #accuracy_score(y_test, y_hls)
    #print("Keras  Accuracy: {}".format(accuracy_score(np.argmax(y_test, axis=1), np.argmax(y_keras, axis=1))))
    print("hls4ml Accuracy: {}".format(accuracy_score(np.argmax(y_test, axis=1), np.argmax(y_hls, axis=1))))

In [ ]:
# Hotfix for crashing, see README
os.environ['LD_PRELOAD'] = '/lib/x86_64-linux-gnu/libudev.so.1'

process_model_durations = []

# Run through every model
for i, model_config in enumerate(model_configs):
    print(f"Processing model {i+1}/{len(model_configs)}: {model_config['description']}")
    print(f"%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%")
    start_time = time.time()
    output_dir = prepare_directory(model_config)
    
    hls_model = compile_model(
        model_config["keras_model_path"],
        output_dir,
        model_config["hls4ml_strategy"],
    )

    evaluate_model(hls_model,model_config["keras_model_path"])
    #print(f"Accuracy after HLS4ML-converting: {}")

    # Create complete bitfile (Vitis Unified-backend) or IP-block (Vitis-backend)
    hls_model.build(
        synth=True,
        bitfile=model_config["hls4ml_generate_bitfile"],
        csim=False # Simulation (CSIM and COSIM) needs input_data_tb and output_data_tb https://fastmachinelearning.org/hls4ml/autodoc/hls4ml.converters.html#hls4ml.converters.convert_from_keras_model
    )
    
    elapsed_time = time.time() - start_time
    process_model_durations.append({
        'model': model_config['description'],
        'time_seconds': elapsed_time,
        'time_minutes': elapsed_time / 60
    })
    print(f"\nModel {i+1} completed in {elapsed_time:.2f} seconds ({elapsed_time/60:.2f} minutes)")


Processing model 1/3: AdaptiveHP acc=0.7084 ebops=864 Distributed Arithmetic
%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%


hls4ml Accuracy: 0.3333333333333333
Processing model 2/3: AdaptiveHP acc=0.7449 ebops=3640 Distributed Arithmetic
%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%
hls4ml Accuracy: 0.3333333333333333
Processing model 3/3: AdaptiveHP acc=0.7573 ebops=11698 Distributed Arithmetic
%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%


KeyboardInterrupt: 

In [ ]:
import pandas as pd

timestamp = time.strftime("%Y%m%d_%H%M%S")
timing_df = pd.DataFrame(process_model_durations)
timing_df = timing_df[["model", "time_seconds", "time_minutes"]]
timing_df.to_csv(f"{model_to_test}/timing_summary_{timestamp}.csv", index=False)

display(timing_df)

total_time = timing_df["time_seconds"].sum()
average_time = timing_df["time_seconds"].mean()
print(f"\nTotal time: {total_time:.2f} seconds ({total_time/60:.2f} minutes)")
print(f"Average time per model: {average_time:.2f} seconds ({average_time/60:.2f} minutes)")

KeyError: "None of [Index(['model', 'time_seconds', 'time_minutes'], dtype='str')] are in the [columns]"

Copy bitfile and hwh from export-directory to directory for onboard-verification.
Make sure the driver and testdata is copied manually.

In [ ]:
import glob
import shutil

target_dir = '../../onboard-verification/pixsplit/DUT'

for model_config in model_configs:
    export_dir = os.path.join(
        os.path.dirname(os.path.abspath(model_config["keras_model_path"])),
        f"hls4ml_prj_{model_config['hls4ml_revision']}",
        'export'
    )
    model_target_dir = os.path.join(
        target_dir,
        f"{model_config['model_revision']}_{model_config['hls4ml_revision']}"
    )
    os.makedirs(model_target_dir, exist_ok=True)

    for extension in ('bit', 'hwh'):
        matches = sorted(glob.glob(os.path.join(export_dir, f'*.{extension}')))
        if not matches:
            print(f"ERROR: No .{extension} file found in {export_dir}")
            continue
        shutil.copy2(matches[0], model_target_dir)
        print(f"Copied {matches[0]} -> {model_target_dir}")

Copied /work/development/pixsplit/pixsplit_hgq2/Training_AdaptiveHP/hls4ml_prj_acc=0.7084_ebops=864_VU_DA_bitfile/export/system.bit -> ../../onboard-verification/pixsplit/DUT/Training_AdaptiveHP_acc=0.7084_ebops=864_VU_DA_bitfile
Copied /work/development/pixsplit/pixsplit_hgq2/Training_AdaptiveHP/hls4ml_prj_acc=0.7084_ebops=864_VU_DA_bitfile/export/system.hwh -> ../../onboard-verification/pixsplit/DUT/Training_AdaptiveHP_acc=0.7084_ebops=864_VU_DA_bitfile
Copied /work/development/pixsplit/pixsplit_hgq2/Training_AdaptiveHP/hls4ml_prj_acc=0.7449_ebops=3640_VU_DA_bitfile/export/system.bit -> ../../onboard-verification/pixsplit/DUT/Training_AdaptiveHP_acc=0.7449_ebops=3640_VU_DA_bitfile
Copied /work/development/pixsplit/pixsplit_hgq2/Training_AdaptiveHP/hls4ml_prj_acc=0.7449_ebops=3640_VU_DA_bitfile/export/system.hwh -> ../../onboard-verification/pixsplit/DUT/Training_AdaptiveHP_acc=0.7449_ebops=3640_VU_DA_bitfile
Copied /work/development/pixsplit/pixsplit_hgq2/Training_AdaptiveHP/hls4ml_p